# EUMETSAT MetOp Data Store Access Notebook

This notebook is a first working example for the MAAP/VEDA/FALCON support task: **use one MetOp product to test EUMETSAT authentication, token creation, product search, and download**.

This notebook defaults to:

`EO:EUM:DAT:METOP:AVHRRL1` — **AVHRR Level 1B - Metop - Global**


## What this notebook does

1. Installs/imports `eumdac`
2. Reads your EUMETSAT consumer key and consumer secret securely
3. Creates a short-lived access token
4. Connects to the EUMETSAT Data Store
5. Selects a MetOp collection
6. Searches a small time window
7. Lists matching products
8. Downloads one product, or optionally one file inside the product

## Notes

- Do **not** hard-code your consumer key or secret in a committed notebook.
- Use environment variables or `getpass` prompts.
- The EUMETSAT Data Store path is not the same as a STAC workflow. This notebook focuses on the Data Store/EUMDAC path first. WEkEO/HDA STAC can be tested separately after the basic token workflow is confirmed.


## 0. How to get the EUMETSAT API key

Before running the notebook:

1. Create/sign in to your [EUMETSAT account](https://user.eumetsat.int/cas/login).
2. Go to the EUMETSAT API key page: [https://api.eumetsat.int/api-key](https://api.eumetsat.int/api-key)
3. Copy the **consumer key** and **consumer secret**.
4. Come back to this notebook and paste them only when prompted.

## 1. Install dependencies

Run this once per clean environment. In MAAP Hub/Jupyter, restart the kernel after installation if imports fail.


In [1]:
%pip install -q eumdac pandas tqdm


Note: you may need to restart the kernel to use updated packages.


## 2. Imports and output folder


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import os
import getpass
import shutil
import json

import pandas as pd
from tqdm.auto import tqdm

import eumdac

DOWNLOAD_DIR = Path("eumetsat_metop_downloads")
DOWNLOAD_DIR.mkdir(exist_ok=True)

print("Download folder:", DOWNLOAD_DIR.resolve())
print("eumdac version:", getattr(eumdac, "__version__", "version not available"))


Download folder: /home/jovyan/eumetsat_metop_downloads
eumdac version: 3.1.1


## 3. Set the test collection

Default product for the first token/search/download test:

EO:EUM:DAT:METOP:AVHRRL1 — AVHRR Level 1B - Metop - Global

This notebook uses the EUMETSAT AVHRR Level 1B - Metop - Global collection as a placeholder while the FALCON team confirms the exact product they need. AVHRR, or Advanced Very High Resolution Radiometer, is an imaging instrument flown on the Metop polar-orbiting satellites. This collection works well for an initial test because it is unambiguously a MetOp product, it is available through the EUMETSAT Data Store, and the collection ID can be swapped out later without touching any of the token, search, or download logic.

Preferred reference:
[EUMETSAT product page — AVHRR Level 1B - Metop - Global](https://user.eumetsat.int/catalogue/EO%3AEUM%3ADAT%3AMETOP%3AAVHRRL1)

Change only this variable later if FALCON asks for a different MetOp product.

In [3]:
COLLECTION_ID = "EO:EUM:DAT:METOP:AVHRRL1"
COLLECTION_NAME_NOTE = "AVHRR Level 1B - Metop - Global"

print("Collection ID:", COLLECTION_ID)
print("Collection note:", COLLECTION_NAME_NOTE)


Collection ID: EO:EUM:DAT:METOP:AVHRRL1
Collection note: AVHRR Level 1B - Metop - Global


## 4. Read credentials securely

This cell checks for environment variables first. If they are not found, it asks you to paste the values.

Environment variable names used here:

- `EUMETSAT_CONSUMER_KEY`
- `EUMETSAT_CONSUMER_SECRET`


In [14]:
def read_secret(env_name: str, prompt: str) -> str:
    value = os.environ.get(env_name)
    if value:
        print(f"Using {env_name} from environment variables.")
        return value
    return getpass.getpass(prompt)

CONSUMER_KEY = read_secret("EUMETSAT_CONSUMER_KEY", "EUMETSAT consumer key: ")
CONSUMER_SECRET = read_secret("EUMETSAT_CONSUMER_SECRET", "EUMETSAT consumer secret: ")

assert CONSUMER_KEY, "Missing consumer key"
assert CONSUMER_SECRET, "Missing consumer secret"

print("Credentials loaded. Secret values are not printed.")


EUMETSAT consumer key:  ········
EUMETSAT consumer secret:  ········


Credentials loaded. Secret values are not printed.


## 5. Create token and connect to the Data Store

`eumdac.AccessToken` exchanges the consumer key/secret for a short-lived access token. Then `eumdac.DataStore` uses that token to interact with the EUMETSAT Data Store.


In [15]:
credentials = (CONSUMER_KEY, CONSUMER_SECRET)

token = eumdac.AccessToken(credentials)
datastore = eumdac.DataStore(token)

print("Access token created successfully.")

# Some eumdac versions expose expiry information; print it if available, but never print the token value.
for attr in ["expiration", "expires", "expires_at"]:
    if hasattr(token, attr):
        try:
            print(f"Token {attr}:", getattr(token, attr))
        except Exception:
            pass


Access token created successfully.
Token expiration: 2026-06-20 16:50:01.131138


## 6. Open the MetOp collection

This should fail fast if the collection ID is wrong or if authentication is not working.


In [16]:
collection = datastore.get_collection(COLLECTION_ID)

print("Selected collection:")
print(collection)


Selected collection:
EO:EUM:DAT:METOP:AVHRRL1


## 7. Search products in a small time window

Start with a small time range so the search is fast. If you get zero products, change the dates.

For a first access test, we only need a few matching products.


In [17]:
# Change these dates if the collection returns no products for this window.
START = datetime(2024, 1, 1, 0, 0, tzinfo=timezone.utc)
END = datetime(2024, 1, 1, 6, 0, tzinfo=timezone.utc)
MAX_RESULTS = 5

print("Searching:")
print("Collection:", COLLECTION_ID)
print("Start:", START)
print("End:", END)
print("Max results:", MAX_RESULTS)

products_iter = collection.search(dtstart=START, dtend=END)

products = []
for product in products_iter:
    products.append(product)
    if len(products) >= MAX_RESULTS:
        break

print(f"Found {len(products)} product(s) in the first {MAX_RESULTS} returned results.")

if not products:
    print("No products found. Try widening START/END or searching without a time filter.")
else:
    for i, product in enumerate(products, start=1):
        print(f"{i}. {product}")


Searching:
Collection: EO:EUM:DAT:METOP:AVHRRL1
Start: 2024-01-01 00:00:00+00:00
End: 2024-01-01 06:00:00+00:00
Max results: 5
Found 5 product(s) in the first 5 returned results.
1. AVHR_xxx_1B_M03_20240101051303Z_20240101065503Z_N_O_20240101065154Z
2. AVHR_xxx_1B_M01_20240101041903Z_20240101060103Z_N_O_20240101050832Z
3. AVHR_xxx_1B_M03_20240101033103Z_20240101051303Z_N_O_20240101050949Z
4. AVHR_xxx_1B_M01_20240101023403Z_20240101041903Z_N_O_20240101032442Z
5. AVHR_xxx_1B_M03_20240101014603Z_20240101033103Z_N_O_20240101032627Z


## 8. Optional fallback: search latest products without a time filter

Run this only if the date-window search above returns no products.


In [18]:
RUN_LATEST_FALLBACK = False

if RUN_LATEST_FALLBACK:
    latest_products = []
    for product in collection.search():
        latest_products.append(product)
        if len(latest_products) >= MAX_RESULTS:
            break

    print(f"Found {len(latest_products)} latest product(s).")
    for i, product in enumerate(latest_products, start=1):
        print(f"{i}. {product}")

    # Use fallback results for the rest of the notebook.
    products = latest_products
else:
    print("Fallback search skipped. Set RUN_LATEST_FALLBACK = True if needed.")


Fallback search skipped. Set RUN_LATEST_FALLBACK = True if needed.


## 9. Put product names into a table

EUMDAC product objects can vary by version/product type, so this helper records safe fields without assuming every metadata field exists.


In [19]:
def safe_getattr(obj, attr):
    try:
        value = getattr(obj, attr)
        if callable(value):
            return None
        return value
    except Exception:
        return None

records = []
for product in products:
    record = {"product_string": str(product)}
    for attr in [
        "id", "identifier", "title", "name", "size", "satellite", "instrument",
        "sensing_start", "sensing_end", "publication", "collection"
    ]:
        value = safe_getattr(product, attr)
        if value is not None:
            record[attr] = str(value)
    records.append(record)

products_df = pd.DataFrame(records)
products_df


,product_string,size,satellite,instrument,sensing_start,sensing_end,collection
0,AVHR_xxx_1B_M03_20240101051303Z_20240101065503...,403950,Metop-C,AVHRR,2024-01-01 05:13:03,2024-01-01 06:55:03,EO:EUM:DAT:METOP:AVHRRL1
1,AVHR_xxx_1B_M01_20240101041903Z_20240101060103...,455329,Metop-B,AVHRR,2024-01-01 04:19:03,2024-01-01 06:01:03,EO:EUM:DAT:METOP:AVHRRL1
2,AVHR_xxx_1B_M03_20240101033103Z_20240101051303...,402704,Metop-C,AVHRR,2024-01-01 03:31:03,2024-01-01 05:13:03,EO:EUM:DAT:METOP:AVHRRL1
3,AVHR_xxx_1B_M01_20240101023403Z_20240101041903...,456826,Metop-B,AVHRR,2024-01-01 02:34:03,2024-01-01 04:19:03,EO:EUM:DAT:METOP:AVHRRL1
4,AVHR_xxx_1B_M03_20240101014603Z_20240101033103...,404912,Metop-C,AVHRR,2024-01-01 01:46:03,2024-01-01 03:31:03,EO:EUM:DAT:METOP:AVHRRL1


## 10. Inspect one product

This checks what files/entries are available inside the product. For some products this may be a ZIP-like product package.


In [20]:
if not products:
    raise RuntimeError("No products available. Run a successful search first.")

selected_product = products[0]
print("Selected product:")
print(selected_product)

entries = []
try:
    entries = list(selected_product.entries)
except Exception as exc:
    print("Could not list product entries:", repr(exc))

print(f"Number of entries detected: {len(entries)}")
for entry in entries[:20]:
    print("-", entry)

if len(entries) > 20:
    print(f"... and {len(entries) - 20} more entries")


Selected product:
AVHR_xxx_1B_M03_20240101051303Z_20240101065503Z_N_O_20240101065154Z
Number of entries detected: 3
- AVHR_xxx_1B_M03_20240101051303Z_20240101065503Z_N_O_20240101065154Z.nat
- EOPMetadata.xml
- manifest.xml


## 11. Download the full selected product

This can be large. The switch is set to `False` by default so the notebook does not accidentally download a huge file.

Set `DOWNLOAD_FULL_PRODUCT = True` when you are ready.


In [21]:
DOWNLOAD_FULL_PRODUCT = False

if DOWNLOAD_FULL_PRODUCT:
    selected_product = products[0]
    print("Downloading full product:", selected_product)

    with selected_product.open() as source:
        output_name = Path(source.name).name
        output_path = DOWNLOAD_DIR / output_name
        with open(output_path, "wb") as destination:
            shutil.copyfileobj(source, destination)

    print("Downloaded to:", output_path.resolve())
    print("File size MB:", round(output_path.stat().st_size / 1_000_000, 2))
else:
    print("Full product download skipped. Set DOWNLOAD_FULL_PRODUCT = True to download.")


Full product download skipped. Set DOWNLOAD_FULL_PRODUCT = True to download.


## 12. Optional: download only one entry/file inside the product

Use this if the product has entries and you want a small metadata file first. This is useful for proving access without downloading the full science product.

The cell tries to pick `manifest.xml` if it exists; otherwise it uses the first detected entry.


In [22]:
DOWNLOAD_SINGLE_ENTRY = False

if DOWNLOAD_SINGLE_ENTRY:
    if not entries:
        raise RuntimeError("No entries were detected for this product. Use the full-product download instead.")

    preferred = None
    for entry in entries:
        if "manifest" in str(entry).lower():
            preferred = entry
            break

    selected_entry = preferred or entries[0]
    print("Downloading entry:", selected_entry)

    with selected_product.open(entry=selected_entry) as source:
        output_name = Path(source.name).name
        output_path = DOWNLOAD_DIR / output_name
        with open(output_path, "wb") as destination:
            shutil.copyfileobj(source, destination)

    print("Downloaded to:", output_path.resolve())
    print("File size MB:", round(output_path.stat().st_size / 1_000_000, 2))
else:
    print("Single-entry download skipped. Set DOWNLOAD_SINGLE_ENTRY = True to download one entry.")


Single-entry download skipped. Set DOWNLOAD_SINGLE_ENTRY = True to download one entry.


## 13. Reusable function for another MetOp collection

When FALCON provides the exact product, change only the `collection_id`, `start`, and `end` values.


In [ ]:
def search_products(collection_id: str, start: datetime, end: datetime, max_results: int = 10):
    # Return up to max_results EUMETSAT Data Store products for a collection and time range.
    col = datastore.get_collection(collection_id)
    results = []
    for product in col.search(dtstart=start, dtend=end):
        results.append(product)
        if len(results) >= max_results:
            break
    return results

# Example use:
# other_products = search_products(
#     collection_id="EO:EUM:DAT:METOP:AVHRRL1",
#     start=datetime(2024, 1, 1, 0, 0, tzinfo=timezone.utc),
#     end=datetime(2024, 1, 1, 6, 0, tzinfo=timezone.utc),
#     max_results=3,
# )
# other_products
